#### Using Form Recognizer

In [1]:
!pip install azure-ai-documentintelligence


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os 
from dotenv import load_dotenv

#get azure key credential
from azure.core.credentials import AzureKeyCredential
#OLD WAY -- from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest

In [3]:
load_dotenv()

True

In [4]:
endpoint = os.getenv("DOCUMENT_ANALYSIS_ENDPOINT")
key = os.getenv("DOCUMENT_ANALYSIS_KEY")

In [5]:
fileuri = "https://github.com/MicrosoftLearning/mslearn-ai-information-extraction/blob/main/Labfiles/03-document-intelligence/prebuilt/sample-invoice/sample-invoice.pdf?raw=true"

In [6]:
client = DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key))

In [9]:
poller = client.begin_analyze_document(
        "prebuilt-invoice",                             # extracts invoice fields (VendorName, InvoiceTotal, ...)
        AnalyzeDocumentRequest(url_source=fileuri),     # a public URL; use bytes_source for a local file
    )                                                   # NOTE: "prebuilt-read" is OCR-only and returns no .documents

In [17]:
poller.status()

'succeeded'

In [18]:
poller.result().documents

[{'docType': 'invoice', 'boundingRegions': [{'pageNumber': 1, 'polygon': [0, 0, 8.5, 0, 8.5, 11, 0, 11]}], 'fields': {'AmountDue': {'type': 'currency', 'valueCurrency': {'currencySymbol': '$', 'amount': 610.0, 'currencyCode': 'USD'}, 'content': '$610.00', 'boundingRegions': [{'pageNumber': 1, 'polygon': [7.3664, 7.8028, 7.9341, 7.8045, 7.9338, 7.9676, 7.3669, 7.9706]}], 'confidence': 0.938, 'spans': [{'offset': 653, 'length': 7}]}, 'BillingAddress': {'type': 'address', 'content': '123 Bill St,\nRedmond WA, 98052', 'boundingRegions': [{'pageNumber': 1, 'polygon': [0.5765, 4.358, 2.0309, 4.3578, 2.0309, 4.7167, 0.5765, 4.7168]}], 'confidence': 0.889, 'spans': [{'offset': 328, 'length': 12}, {'offset': 370, 'length': 17}], 'valueAddress': {'houseNumber': '123', 'road': 'Bill St', 'postalCode': '98052', 'city': 'Redmond', 'state': 'WA', 'streetAddress': '123 Bill St'}}, 'BillingAddressRecipient': {'type': 'string', 'valueString': 'Microsoft Finance', 'content': 'Microsoft Finance', 'boundi

In [19]:
poller.result().documents[0].fields.keys()

dict_keys(['AmountDue', 'BillingAddress', 'BillingAddressRecipient', 'CustomerAddress', 'CustomerAddressRecipient', 'CustomerId', 'CustomerName', 'DueDate', 'InvoiceDate', 'InvoiceId', 'InvoiceTotal', 'Items', 'PreviousUnpaidBalance', 'PurchaseOrder', 'RemittanceAddress', 'RemittanceAddressRecipient', 'ServiceAddress', 'ServiceAddressRecipient', 'ShippingAddress', 'ShippingAddressRecipient', 'SubTotal', 'TaxDetails', 'TotalTax', 'VendorAddress', 'VendorAddressRecipient', 'VendorName'])

### Document Intelligence

In [20]:
!pip install azure-ai-documentintelligence


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
from azure.ai.documentintelligence import DocumentIntelligenceClient as di
from azure.ai.documentintelligence.models import DocumentAnalysisFeature, AnalyzeResult

In [22]:
client = di(endpoint=endpoint, credential=AzureKeyCredential(key))

In [23]:
# Helper: is a word fully contained within any of the given text spans?
def _in_span(word, spans):
    for span in spans:
        # a word belongs to the span if its offset range sits inside the span's offset range
        if word.span.offset >= span.offset and (word.span.offset + word.span.length) <= (span.offset + span.length):
            return True
    return False

# Helper: turn a list of bounding regions into a readable "Page #n: [x, y], ..." string
def _format_bounding_region(bounding_regions):
    if not bounding_regions:                      # no location info available
        return "N/A"
    # one entry per region, showing its page number and polygon corners
    return ", ".join(
        f"Page #{region.page_number}: {_format_polygon(region.polygon)}" for region in bounding_regions
    )

# Helper: format a flat [x1, y1, x2, y2, ...] polygon into readable [x, y] coordinate pairs
def _format_polygon(polygon):
    if not polygon:                               # no polygon points available
        return "N/A"
    # step through the flat list two values at a time to rebuild each (x, y) corner
    return ", ".join([f"[{polygon[i]}, {polygon[i + 1]}]" for i in range(0, len(polygon), 2)])

In [44]:
# Open the local invoice PDF in binary mode and send it to Document Intelligence
with open("invoice-1234.pdf", "rb") as f:
    poller = client.begin_analyze_document(
        "prebuilt-layout",                            # layout model: text, lines, tables, structure
        f,                                            # the raw file bytes (not a URL this time)
        features=[DocumentAnalysisFeature.KEY_VALUE_PAIRS],  # also extract key/value pairs
        content_type="application/octet-stream",      # tell the service we're sending raw bytes
    )                                                 # returns a poller for the async analysis job

In [45]:
poller.status()   # check the current state of the async job (e.g. "running" / "succeeded")

'succeeded'

In [46]:
result = poller.result()   # block until the job finishes, then get the AnalyzeResult

In [47]:
result.styles

[{'confidence': 1, 'spans': [{'offset': 357, 'length': 16}], 'isHandwritten': True}]

In [48]:
# Detect and print any handwritten text the model found
if result.styles:                                 # styles describe text characteristics (e.g. handwriting)
    for style in result.styles:
        if style.is_handwritten:                  # only care about handwritten runs here
            print("Document contains handwritten content: ")
            # each span points to a slice of result.content; slice it out and print the text
            print(",".join([result.content[span.offset : span.offset + span.length] for span in style.spans]))


Document contains handwritten content: 
Brandon was here


In [49]:
# Print every key/value pair (e.g. "Invoice No:" -> "1234") with its location on the page
print("----Key-value pairs found in document----")
if result.key_value_pairs:                        # only present because we requested KEY_VALUE_PAIRS
    for kv_pair in result.key_value_pairs:
        if kv_pair.key:                           # the label/field name side of the pair
            print(
                f"Key '{kv_pair.key.content}' found within "
                f"'{_format_bounding_region(kv_pair.key.bounding_regions)}' bounding regions"
            )
        if kv_pair.value:                         # the corresponding value side of the pair
            print(
                f"Value '{kv_pair.value.content}' found within "
                f"'{_format_bounding_region(kv_pair.value.bounding_regions)}' bounding regions\n"
            )


----Key-value pairs found in document----
Key 'Invoice No:' found within 'Page #1: [5.4439, 0.8821], [6.1979, 0.8817], [6.198, 1.0234], [5.444, 1.0238]' bounding regions
Value '1234' found within 'Page #1: [6.4025, 0.8847], [6.7292, 0.884], [6.7294, 1.0162], [6.4028, 1.0169]' bounding regions

Key 'Tel:' found within 'Page #1: [0.8546, 1.2843], [1.0896, 1.2845], [1.0895, 1.4315], [0.8545, 1.4313]' bounding regions
Value '555 123-4567' found within 'Page #1: [1.1375, 1.2826], [2.063, 1.2838], [2.0628, 1.4327], [1.1373, 1.4316]' bounding regions

Key 'Date:' found within 'Page #1: [5.4514, 1.2671], [5.811, 1.2682], [5.8105, 1.4001], [5.451, 1.3989]' bounding regions
Value '03/07/2025' found within 'Page #1: [6.3929, 1.2543], [7.1719, 1.2532], [7.1721, 1.4107], [6.3932, 1.4119]' bounding regions

Key 'Customer Name:' found within 'Page #1: [1.3568, 2.0369], [2.4681, 2.0372], [2.4681, 2.179], [1.3568, 2.1787]' bounding regions
Value 'John Smith' found within 'Page #1: [2.7343, 2.031], [3.4

In [ ]:
# Walk every page and print its dimensions, each line of text, and any selection marks
for page in result.pages:
    print(f"----Analyzing document from page #{page.page_number}----")
    print(f"Page has width: {page.width} and height: {page.height}, measured with unit: {page.unit}")

    if page.lines:                                    # each detected line of text on the page
        for line_idx, line in enumerate(page.lines):
            words = []
            if page.words:
                for word in page.words:
                    #print(f"......Word '{word.content}' has a confidence of {word.confidence}")
                    if _in_span(word, line.spans):    # keep only the words belonging to this line
                        words.append(word)
            print(
                f"...Line #{line_idx} has {len(words)} words and text '{line.content}' within "
                f"bounding polygon '{_format_polygon(line.polygon)}'"
            )

    if page.selection_marks:                          # checkboxes / radio buttons and their state
        for selection_mark in page.selection_marks:
            print(
                f"Selection mark is '{selection_mark.state}' within bounding polygon "
                f"'{_format_polygon(selection_mark.polygon)}' and has a confidence of "
                f"{selection_mark.confidence}"
            )